In [ ]:
input_data = None
input_model = None
datanames = None
util = None
display_util = None
targetpop_data = None
output_data = None

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline

from IPython.display import Markdown
import numpy as np
from joblib import Parallel, delayed
import multiprocessing

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")


%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import collist, fcount, rule_setup  # noqa: E402,


def singeval(col):
    col = col.value_counts().index
    if col.size == 0:
        return np.nan
    else:
        assert col.size == 1
        return col[0]


def joinvals(col):
    if col.dropna().size == 0:
        return np.nan
    return ";;".join(col.astype("str"))


def temp_func(func, name, group):
    return group.agg(func), name


def getaggfuns(df):
    retlist = Parallel(n_jobs=int(multiprocessing.cpu_count() / 2))(
        delayed(temp_func)(pd.Series.nunique, name, group)
        for name, group in df.groupby(df.index.names, dropna=False)
    )
    idx = pd.MultiIndex.from_tuples((g for agg, g in retlist), names=df.index.names)
    counts = pd.DataFrame((agg for agg, g in retlist), index=idx)
    multicol = counts.columns[counts.max(axis=0) > 1]
    if multicol.size != 0:
        display(
            Markdown(
                f"""Because the columns {collist(multicol)} have multiple values, they are collapsed with ';;' as a seperator."""
            )
        )
    return {col: joinvals if col in multicol else singeval for col in df.columns}


def aggts(df):
    dfGrouped = df.groupby(df.index.names, dropna=False)
    funs = getaggfuns(df)
    retlist = Parallel(n_jobs=int(multiprocessing.cpu_count() / 2))(
        delayed(temp_func)(funs, name, group) for name, group in dfGrouped
    )
    idx = pd.MultiIndex.from_tuples((g for agg, g in retlist), names=df.index.names)
    return pd.DataFrame((agg for agg, g in retlist), index=idx)

# Creation of consolidated dataset

To make the usage of the dataset available to the user in a single dataset, all other datasets are joined to a single dataset. Long form data (time series) are collapsed to sequences, which can be "exploded" back to a time series. This means each transplantation is represented by a single row. Dates are made relative to the transplantation.

In [ ]:
data = pd.Series(input_data.split(" "))
models = input_model.split(" ")
names = pd.Series(datanames.split(" "))
targetpop = pd.read_parquet(targetpop_data)

display(Markdown("### Input Data Summary"))
display(Markdown(f"- **Number of intermediate datasets**: {len(data)}"))
display(Markdown(f"- **Target population size**: {len(targetpop)} transplants"))

## Wide Data

The wide data frames with no long form data can be simply read and joined. 

### empfaenger

In [ ]:
emf = pd.read_parquet(data[names == "empfaenger"].iloc[0])
display(Markdown(f"**Before merge**: {emf.shape[0]} rows × {emf.shape[1]} columns"))
emf = pd.merge(
    emf,
    targetpop,
    left_index=True,
    right_on="recipient_et_id_et",
    how="inner",
    validate="1:1",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge with target population**: {emf.shape[0]} rows × {emf.shape[1]} columns"
    )
)
datecols = emf.columns[emf.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    emf[col] -= emf.index.get_level_values("recipient_op_date")
display(Markdown(f"**Final shape**: {emf.shape[0]} rows × {emf.shape[1]} columns"))

### organ_entnahme_niere

In [ ]:
orgent = pd.read_parquet(data[names == "organ_entnahme_niere"].iloc[0])
display(
    Markdown(f"**Before merge**: {orgent.shape[0]} rows × {orgent.shape[1]} columns")
)
orgent = pd.merge(
    orgent,
    targetpop,
    left_index=True,
    right_on=["transplant_et_id", "donor_et_id_et"],
    how="inner",
    validate="1:1",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge with target population**: {orgent.shape[0]} rows × {orgent.shape[1]} columns"
    )
)
datecols = orgent.columns[orgent.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    orgent[col] -= orgent.index.get_level_values("recipient_op_date")
display(
    Markdown(f"**Final shape**: {orgent.shape[0]} rows × {orgent.shape[1]} columns")
)

### spender_postmortem

In [ ]:
spendpost = pd.read_parquet(data[names == "spender_postmortem"].iloc[0])
display(
    Markdown(
        f"**Before merge**: {spendpost.shape[0]} rows × {spendpost.shape[1]} columns"
    )
)
spendpost = pd.merge(
    spendpost,
    targetpop,
    left_index=True,
    right_on=["donor_et_id_et"],
    how="inner",
    validate="1:m",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge with target population**: {spendpost.shape[0]} rows × {spendpost.shape[1]} columns"
    )
)
datecols = spendpost.columns[spendpost.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    spendpost[col] -= spendpost.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Final shape**: {spendpost.shape[0]} rows × {spendpost.shape[1]} columns"
    )
)

## Long Data

In each dataset not all columns contain long data, therefore only the columns with multiple values are "collapsed."

### empfaenger_dringlichkeit

In [ ]:
empdring = pd.read_parquet(data[names == "empfaenger_dringlichkeit"].iloc[0])
display(
    Markdown(
        f"**Before merge**: {empdring.shape[0]} rows × {empdring.shape[1]} columns"
    )
)
empdring = pd.merge(
    empdring,
    targetpop,
    left_index=True,
    right_on=["recipient_et_id_et"],
    how="inner",
    validate="m:1",
).set_index(list(targetpop.columns))
display(
    Markdown(f"**After merge**: {empdring.shape[0]} rows × {empdring.shape[1]} columns")
)
datecols = empdring.columns[empdring.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    empdring[col] -= empdring.index.get_level_values("recipient_op_date")
sel = empdring["date"] < 14
display(
    Markdown(
        f"{fcount((~sel).sum(), sel.shape[0])} entries of waiting list changes are removed, as they are registered later than 14 days."
    )
)
empdring = empdring[sel].copy()
display(
    Markdown(
        f"**Before aggregation**: {empdring.shape[0]} rows × {empdring.shape[1]} columns"
    )
)
empdring = aggts(empdring)
display(
    Markdown(
        f"**After aggregation**: {empdring.shape[0]} rows × {empdring.shape[1]} columns"
    )
)

### empfaenger_immunologie

In this dataset we split by type of the result, resulting in multiple datasets which are joined to the consolidated dataset.

In [ ]:
empimm = pd.read_parquet(data[names == "empfaenger_immunologie"].iloc[0])
display(
    Markdown(f"**Before merge**: {empimm.shape[0]} rows × {empimm.shape[1]} columns")
)
empimm = pd.merge(
    empimm,
    targetpop,
    left_index=True,
    right_on=["recipient_et_id_et"],
    how="inner",
    validate="m:1",
).set_index(list(targetpop.columns))
display(
    Markdown(f"**After merge**: {empimm.shape[0]} rows × {empimm.shape[1]} columns")
)
datecols = empimm.columns[empimm.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    empimm[col] -= empimm.index.get_level_values("recipient_op_date")
sel = empimm["sampling_date"] <= 0
display(
    Markdown(
        f"{fcount((~sel).sum(), sel.shape[0])} entries of lab results are removed, as they used samples after the transplant."
    )
)
empimm = empimm[sel].copy()

empimmsplit = {}
# Only latest result
for testtype in ["Unacceptable Test", "Acceptable Test", "HLA Typing"]:
    empimmsplit[testtype] = (
        empimm[empimm["result_type"] == testtype]
        .dropna(how="all", axis=1)
        .drop(columns="result_type")
    )
    before = empimmsplit[testtype].shape[0]
    empimmsplit[testtype] = (
        empimmsplit[testtype].groupby(empimm.index.names, dropna=False).tail(1)
    )
    after = empimmsplit[testtype].shape[0]
    display(
        Markdown(
            f"For the test type '{testtype}' only the latest result was kept and it became its own dataset. This removed {fcount(after, before)} results. Final shape: {empimmsplit[testtype].shape[0]} rows × {empimmsplit[testtype].shape[1]} columns"
        )
    )

# Split by DTT used or not (or unknown)
antibodyscreen = (
    empimm[empimm["result_type"] == "Antibody Screening"]
    .dropna(how="all", axis=1)
    .drop(columns="result_type")
)
for val, iloc in antibodyscreen.groupby("dtt_crossmatch", dropna=False).indices.items():
    current = antibodyscreen.iloc[iloc, :]
    name = f"Antibody Screening: dtt_crossmatch={val}"
    display(
        Markdown(
            f"The results with {name} - shape: {current.shape[0]} rows × {current.shape[1]} columns"
        )
    )
    empimmsplit[name] = aggts(current).drop(columns="dtt_crossmatch")
    display(
        Markdown(
            f"  After aggregation: {empimmsplit[name].shape[0]} rows × {empimmsplit[name].shape[1]} columns"
        )
    )
newnames = (
    pd.Series(empimmsplit.keys())
    .str.lower()
    .str.replace("[^a-zA-Z0-9_]", "_", regex=True)
)
display(Markdown(f"The following (sub)datasets were produced: {', '.join(newnames)}"))
for k, new in zip(empimmsplit.keys(), newnames):
    empimmsplit[k].columns = [new + "_" + col for col in empimmsplit[k].columns]

### empfaenger_virologie

In [ ]:
empvir = pd.read_parquet(data[names == "empfaenger_virologie"].iloc[0])
display(
    Markdown(f"**Before merge**: {empvir.shape[0]} rows × {empvir.shape[1]} columns")
)
empvir = pd.merge(
    empvir,
    targetpop,
    left_index=True,
    right_on=["recipient_et_id_et"],
    how="inner",
    validate="m:1",
).set_index(list(targetpop.columns))
display(
    Markdown(f"**After merge**: {empvir.shape[0]} rows × {empvir.shape[1]} columns")
)
datecols = empvir.columns[empvir.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    empvir[col] -= empvir.index.get_level_values("recipient_op_date")
sel = empvir["sampling_date"] <= 0
display(
    Markdown(
        f"{fcount((~sel).sum(), sel.shape[0])} entries of lab results are removed, as they used samples after the transplant."
    )
)
empvir = empvir[sel].copy()
before = empvir.shape[0]
empvir = empvir.groupby(empvir.index.names, dropna=False).tail(1).copy()
after = empvir.shape[0]
display(
    Markdown(
        f"Only the latest results were kept. This removed {fcount(before - after, before)} results. Final shape: {empvir.shape[0]} rows × {empvir.shape[1]} columns"
    )
)
assert not empvir.index.duplicated().any()

### followup_niere

In [ ]:
follown = pd.read_parquet(data[names == "followup_niere"].iloc[0])
display(
    Markdown(f"**Before merge**: {follown.shape[0]} rows × {follown.shape[1]} columns")
)
follown.index = follown.index.droplevel("transplant_et_id")
follown = pd.merge(
    follown,
    targetpop,
    left_index=True,
    right_on=["recipient_et_id_et"],
    how="inner",
    validate="m:1",
).set_index(list(targetpop.columns))
display(
    Markdown(f"**After merge**: {follown.shape[0]} rows × {follown.shape[1]} columns")
)
datecols = follown.columns[follown.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    follown[col] -= follown.index.get_level_values("recipient_op_date")  #
display(
    Markdown(
        f"**Before aggregation**: {follown.shape[0]} rows × {follown.shape[1]} columns"
    )
)
follown = aggts(follown)
display(
    Markdown(
        f"**After aggregation**: {follown.shape[0]} rows × {follown.shape[1]} columns"
    )
)

### followup_niere_medikation

In [ ]:
follownmed = pd.read_parquet(data[names == "followup_niere_medikation"].iloc[0])
display(
    Markdown(
        f"**Before merge**: {follownmed.shape[0]} rows × {follownmed.shape[1]} columns"
    )
)
follownmed = pd.merge(
    follownmed,
    targetpop,
    left_index=True,
    right_on=["transplant_et_id"],
    how="inner",
    validate="m:1",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {follownmed.shape[0]} rows × {follownmed.shape[1]} columns"
    )
)
datecols = follownmed.columns[follownmed.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    follownmed[col] -= follownmed.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {follownmed.shape[0]} rows × {follownmed.shape[1]} columns"
    )
)
follownmed = aggts(follownmed)
display(
    Markdown(
        f"**After aggregation**: {follownmed.shape[0]} rows × {follownmed.shape[1]} columns"
    )
)

### spender_postmortem_diagnosen

In [ ]:
spendpost_dia = pd.read_parquet(data[names == "spender_postmortem_diagnosen"].iloc[0])
display(
    Markdown(
        f"**Before merge**: {spendpost_dia.shape[0]} rows × {spendpost_dia.shape[1]} columns"
    )
)
spendpost_dia = pd.merge(
    spendpost_dia,
    targetpop,
    left_index=True,
    right_on=["donor_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {spendpost_dia.shape[0]} rows × {spendpost_dia.shape[1]} columns"
    )
)
datecols = spendpost_dia.columns[spendpost_dia.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    spendpost_dia[col] -= spendpost_dia.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {spendpost_dia.shape[0]} rows × {spendpost_dia.shape[1]} columns"
    )
)
spendpost_dia = aggts(spendpost_dia)
display(
    Markdown(
        f"**After aggregation**: {spendpost_dia.shape[0]} rows × {spendpost_dia.shape[1]} columns"
    )
)

### spender_postmortem_labor_blutgase

In [ ]:
spendpost_bg = pd.read_parquet(
    data[names == "spender_postmortem_labor_blutgase"].iloc[0]
)
display(
    Markdown(
        f"**Before merge**: {spendpost_bg.shape[0]} rows × {spendpost_bg.shape[1]} columns"
    )
)
spendpost_bg = pd.merge(
    spendpost_bg,
    targetpop,
    left_index=True,
    right_on=["donor_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {spendpost_bg.shape[0]} rows × {spendpost_bg.shape[1]} columns"
    )
)
datecols = spendpost_bg.columns[spendpost_bg.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    spendpost_bg[col] -= spendpost_bg.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {spendpost_bg.shape[0]} rows × {spendpost_bg.shape[1]} columns"
    )
)
spendpost_bg = aggts(spendpost_bg)
display(
    Markdown(
        f"**After aggregation**: {spendpost_bg.shape[0]} rows × {spendpost_bg.shape[1]} columns"
    )
)

### spender_postmortem_labor_blutgruppe

In [ ]:
spendpost_blg = pd.read_parquet(
    data[names == "spender_postmortem_labor_blutgruppe"].iloc[0]
)
display(
    Markdown(
        f"**Before merge**: {spendpost_blg.shape[0]} rows × {spendpost_blg.shape[1]} columns"
    )
)
spendpost_blg = pd.merge(
    spendpost_blg,
    targetpop,
    left_index=True,
    right_on=["donor_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {spendpost_blg.shape[0]} rows × {spendpost_blg.shape[1]} columns"
    )
)
datecols = spendpost_blg.columns[spendpost_blg.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    spendpost_blg[col] -= spendpost_blg.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {spendpost_blg.shape[0]} rows × {spendpost_blg.shape[1]} columns"
    )
)
spendpost_blg = aggts(spendpost_blg)
display(
    Markdown(
        f"**After aggregation**: {spendpost_blg.shape[0]} rows × {spendpost_blg.shape[1]} columns"
    )
)

### spender_postmortem_labor_hla

In [ ]:
spendpost_hla = pd.read_parquet(data[names == "spender_postmortem_labor_hla"].iloc[0])
display(
    Markdown(
        f"**Before merge**: {spendpost_hla.shape[0]} rows × {spendpost_hla.shape[1]} columns"
    )
)
spendpost_hla = pd.merge(
    spendpost_hla,
    targetpop,
    left_index=True,
    right_on=["donor_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {spendpost_hla.shape[0]} rows × {spendpost_hla.shape[1]} columns"
    )
)
datecols = spendpost_hla.columns[spendpost_hla.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    spendpost_hla[col] -= spendpost_hla.index.get_level_values("recipient_op_date")


before = spendpost_hla.shape[0]
spendpost_hla = spendpost_hla.groupby(spendpost_hla.index.names, dropna=False).tail(1)
after = spendpost_hla.shape[0]
display(
    Markdown(
        f"Only the latest results were kept. This removed {fcount(before - after, before)} results. Final shape: {spendpost_hla.shape[0]} rows × {spendpost_hla.shape[1]} columns"
    )
)

### spender_postmortem_labor_klinische_chemie

In [ ]:
spendpost_kc = pd.read_parquet(
    data[names == "spender_postmortem_labor_klinische_chemie"].iloc[0]
)
display(
    Markdown(
        f"**Before merge**: {spendpost_kc.shape[0]} rows × {spendpost_kc.shape[1]} columns"
    )
)
spendpost_kc = pd.merge(
    spendpost_kc,
    targetpop,
    left_index=True,
    right_on=["donor_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {spendpost_kc.shape[0]} rows × {spendpost_kc.shape[1]} columns"
    )
)
datecols = spendpost_kc.columns[spendpost_kc.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    spendpost_kc[col] -= spendpost_kc.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {spendpost_kc.shape[0]} rows × {spendpost_kc.shape[1]} columns"
    )
)
spendpost_kc = aggts(spendpost_kc)
display(
    Markdown(
        f"**After aggregation**: {spendpost_kc.shape[0]} rows × {spendpost_kc.shape[1]} columns"
    )
)

### spender_postmortem_labor_mikrobiologie

In [ ]:
spendpost_mb = pd.read_parquet(
    data[names == "spender_postmortem_labor_mikrobiologie"].iloc[0]
)
display(
    Markdown(
        f"**Before merge**: {spendpost_mb.shape[0]} rows × {spendpost_mb.shape[1]} columns"
    )
)
spendpost_mb = pd.merge(
    spendpost_mb,
    targetpop,
    left_index=True,
    right_on=["donor_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {spendpost_mb.shape[0]} rows × {spendpost_mb.shape[1]} columns"
    )
)
datecols = spendpost_mb.columns[spendpost_mb.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    spendpost_mb[col] -= spendpost_mb.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {spendpost_mb.shape[0]} rows × {spendpost_mb.shape[1]} columns"
    )
)
spendpost_mb = aggts(spendpost_mb)
display(
    Markdown(
        f"**After aggregation**: {spendpost_mb.shape[0]} rows × {spendpost_mb.shape[1]} columns"
    )
)

### spender_postmortem_labor_pathologie

In [ ]:
spendpost_pl = pd.read_parquet(
    data[names == "spender_postmortem_labor_pathologie"].iloc[0]
)
display(
    Markdown(
        f"**Before merge**: {spendpost_pl.shape[0]} rows × {spendpost_pl.shape[1]} columns"
    )
)
spendpost_pl = pd.merge(
    spendpost_pl,
    targetpop,
    left_index=True,
    right_on=["donor_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {spendpost_pl.shape[0]} rows × {spendpost_pl.shape[1]} columns"
    )
)
datecols = spendpost_pl.columns[spendpost_pl.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    spendpost_pl[col] -= spendpost_pl.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {spendpost_pl.shape[0]} rows × {spendpost_pl.shape[1]} columns"
    )
)
spendpost_pl = aggts(spendpost_pl)
display(
    Markdown(
        f"**After aggregation**: {spendpost_pl.shape[0]} rows × {spendpost_pl.shape[1]} columns"
    )
)

### spender_postmortem_labor_toxikologie

In [ ]:
spendpost_tox = pd.read_parquet(
    data[names == "spender_postmortem_labor_toxikologie"].iloc[0]
)
display(
    Markdown(
        f"**Before merge**: {spendpost_tox.shape[0]} rows × {spendpost_tox.shape[1]} columns"
    )
)
spendpost_tox = pd.merge(
    spendpost_tox,
    targetpop,
    left_index=True,
    right_on=["donor_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {spendpost_tox.shape[0]} rows × {spendpost_tox.shape[1]} columns"
    )
)
datecols = spendpost_tox.columns[spendpost_tox.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    spendpost_tox[col] -= spendpost_tox.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {spendpost_tox.shape[0]} rows × {spendpost_tox.shape[1]} columns"
    )
)
spendpost_tox = aggts(spendpost_tox)
display(
    Markdown(
        f"**After aggregation**: {spendpost_tox.shape[0]} rows × {spendpost_tox.shape[1]} columns"
    )
)

### spender_postmortem_labor_urin

In [ ]:
spendpost_ur = pd.read_parquet(data[names == "spender_postmortem_labor_urin"].iloc[0])
display(
    Markdown(
        f"**Before merge**: {spendpost_ur.shape[0]} rows × {spendpost_ur.shape[1]} columns"
    )
)
spendpost_ur = pd.merge(
    spendpost_ur,
    targetpop,
    left_index=True,
    right_on=["donor_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {spendpost_ur.shape[0]} rows × {spendpost_ur.shape[1]} columns"
    )
)
datecols = spendpost_ur.columns[spendpost_ur.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    spendpost_ur[col] -= spendpost_ur.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {spendpost_ur.shape[0]} rows × {spendpost_ur.shape[1]} columns"
    )
)
spendpost_ur = aggts(spendpost_ur)
display(
    Markdown(
        f"**After aggregation**: {spendpost_ur.shape[0]} rows × {spendpost_ur.shape[1]} columns"
    )
)

### spender_postmortem_labor_virologie

In [ ]:
spendpost_vir = pd.read_parquet(
    data[names == "spender_postmortem_labor_virologie"].iloc[0]
)
display(
    Markdown(
        f"**Before merge**: {spendpost_vir.shape[0]} rows × {spendpost_vir.shape[1]} columns"
    )
)
spendpost_vir = pd.merge(
    spendpost_vir,
    targetpop,
    left_index=True,
    right_on=["donor_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {spendpost_vir.shape[0]} rows × {spendpost_vir.shape[1]} columns"
    )
)
datecols = spendpost_vir.columns[spendpost_vir.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    spendpost_vir[col] -= spendpost_vir.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {spendpost_vir.shape[0]} rows × {spendpost_vir.shape[1]} columns"
    )
)
spendpost_vir = aggts(spendpost_vir)
display(
    Markdown(
        f"**After aggregation**: {spendpost_vir.shape[0]} rows × {spendpost_vir.shape[1]} columns"
    )
)

### spender_postmortem_medikation

In [ ]:
spendpost_med = pd.read_parquet(data[names == "spender_postmortem_medikation"].iloc[0])
display(
    Markdown(
        f"**Before merge**: {spendpost_med.shape[0]} rows × {spendpost_med.shape[1]} columns"
    )
)
spendpost_med = pd.merge(
    spendpost_med,
    targetpop,
    left_index=True,
    right_on=["donor_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {spendpost_med.shape[0]} rows × {spendpost_med.shape[1]} columns"
    )
)
datecols = spendpost_med.columns[spendpost_med.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    spendpost_med[col] -= spendpost_med.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {spendpost_med.shape[0]} rows × {spendpost_med.shape[1]} columns"
    )
)
spendpost_med = aggts(spendpost_med)
display(
    Markdown(
        f"**After aggregation**: {spendpost_med.shape[0]} rows × {spendpost_med.shape[1]} columns"
    )
)

### spender_postmortem_monitoring

In [ ]:
spendpost_mon = pd.read_parquet(data[names == "spender_postmortem_monitoring"].iloc[0])
display(
    Markdown(
        f"**Before merge**: {spendpost_mon.shape[0]} rows × {spendpost_mon.shape[1]} columns"
    )
)
spendpost_mon = pd.merge(
    spendpost_mon,
    targetpop,
    left_index=True,
    right_on=["donor_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {spendpost_mon.shape[0]} rows × {spendpost_mon.shape[1]} columns"
    )
)
datecols = spendpost_mon.columns[spendpost_mon.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    spendpost_mon[col] -= spendpost_mon.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {spendpost_mon.shape[0]} rows × {spendpost_mon.shape[1]} columns"
    )
)
spendpost_mon = aggts(spendpost_mon)
display(
    Markdown(
        f"**After aggregation**: {spendpost_mon.shape[0]} rows × {spendpost_mon.shape[1]} columns"
    )
)

### spender_postmortem_untersuchungen

In [ ]:
spendpost_exam = pd.read_parquet(
    data[names == "spender_postmortem_untersuchungen"].iloc[0]
)
display(
    Markdown(
        f"**Before merge**: {spendpost_exam.shape[0]} rows × {spendpost_exam.shape[1]} columns"
    )
)
spendpost_exam = pd.merge(
    spendpost_exam,
    targetpop,
    left_index=True,
    right_on=["donor_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {spendpost_exam.shape[0]} rows × {spendpost_exam.shape[1]} columns"
    )
)
datecols = spendpost_exam.columns[spendpost_exam.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    spendpost_exam[col] -= spendpost_exam.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {spendpost_exam.shape[0]} rows × {spendpost_exam.shape[1]} columns"
    )
)
spendpost_exam = aggts(spendpost_exam)
display(
    Markdown(
        f"**After aggregation**: {spendpost_exam.shape[0]} rows × {spendpost_exam.shape[1]} columns"
    )
)

### transplantation_postop_untersuchung

In [ ]:
transpost_exam = pd.read_parquet(
    data[names == "transplantation_postop_untersuchung"].iloc[0]
)
display(
    Markdown(
        f"**Before merge**: {transpost_exam.shape[0]} rows × {transpost_exam.shape[1]} columns"
    )
)
transpost_exam = pd.merge(
    transpost_exam,
    targetpop,
    left_index=True,
    right_on=["transplant_et_id"],
    how="inner",
    validate="m:1",
).set_index(list(targetpop.columns))
display(
    Markdown(
        f"**After merge**: {transpost_exam.shape[0]} rows × {transpost_exam.shape[1]} columns"
    )
)
datecols = transpost_exam.columns[transpost_exam.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    transpost_exam[col] -= transpost_exam.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {transpost_exam.shape[0]} rows × {transpost_exam.shape[1]} columns"
    )
)
transpost_exam = aggts(transpost_exam)
display(
    Markdown(
        f"**After aggregation**: {transpost_exam.shape[0]} rows × {transpost_exam.shape[1]} columns"
    )
)

### warteliste_niere

In [ ]:
waitn = pd.read_parquet(data[names == "warteliste_niere"].iloc[0])
display(Markdown(f"**Before merge**: {waitn.shape[0]} rows × {waitn.shape[1]} columns"))
waitn = pd.merge(
    waitn,
    targetpop,
    left_index=True,
    right_on=["recipient_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(Markdown(f"**After merge**: {waitn.shape[0]} rows × {waitn.shape[1]} columns"))
datecols = waitn.columns[waitn.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    waitn[col] -= waitn.index.get_level_values("recipient_op_date")

sel = (waitn["date"] < 14) & (waitn["dialysis_start_date"] <= 14)
display(
    Markdown(
        f"{fcount((~sel).sum(), sel.shape[0])} entries of waiting list information was removed, as it either was registered 14 after the transplantation or dialysis was reported after the transplantation (ET and IQTIG filtering)."
    )
)
waitn = waitn[sel].copy()

display(
    Markdown(
        f"**Before aggregation**: {waitn.shape[0]} rows × {waitn.shape[1]} columns"
    )
)
waitn = aggts(waitn)
display(
    Markdown(f"**After aggregation**: {waitn.shape[0]} rows × {waitn.shape[1]} columns")
)

### transplantation

In [ ]:
trans = pd.read_parquet(data[names == "transplantation"].iloc[0])
display(
    Markdown(f"**Before index drop**: {trans.shape[0]} rows × {trans.shape[1]} columns")
)
trans.index = trans.index.droplevel(["transplant_et_id", "donor_et_id_et"])
trans = pd.merge(
    trans,
    targetpop,
    left_index=True,
    right_on=["recipient_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(Markdown(f"**After merge**: {trans.shape[0]} rows × {trans.shape[1]} columns"))
datecols = trans.columns[trans.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    trans[col] -= trans.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {trans.shape[0]} rows × {trans.shape[1]} columns"
    )
)
trans = aggts(trans)
display(
    Markdown(f"**After aggregation**: {trans.shape[0]} rows × {trans.shape[1]} columns")
)

## spender_postmortem_labor_crossmatch

In [ ]:
xmatch = pd.read_parquet(data[names == "spender_postmortem_labor_crossmatch"].iloc[0])
display(
    Markdown(f"**Before merge**: {xmatch.shape[0]} rows × {xmatch.shape[1]} columns")
)
xmatch = pd.merge(
    xmatch,
    targetpop,
    on=["recipient_et_id_et", "donor_et_id_et"],
    how="inner",
    validate="m:m",
).set_index(list(targetpop.columns))
display(
    Markdown(f"**After merge**: {xmatch.shape[0]} rows × {xmatch.shape[1]} columns")
)
datecols = xmatch.columns[xmatch.columns.str.contains("date")]
display(
    Markdown(
        f"""The columns {collist(datecols)} are made relative to the operation date.
"""
    )
)
for col in datecols:
    xmatch[col] -= xmatch.index.get_level_values("recipient_op_date")
display(
    Markdown(
        f"**Before aggregation**: {xmatch.shape[0]} rows × {xmatch.shape[1]} columns"
    )
)
xmatch = aggts(xmatch)
display(
    Markdown(
        f"**After aggregation**: {xmatch.shape[0]} rows × {xmatch.shape[1]} columns"
    )
)

## Joining everything

All datasets are joined.

In [ ]:
display(Markdown("### Joining all datasets"))
consolidated = (
    pd.concat(
        [
            emf,
            orgent,
            spendpost,
            empdring,
            *empimmsplit.values(),
            empvir,
            follown,
            follownmed,
            spendpost_dia,
            spendpost_bg,
            spendpost_hla,
            spendpost_kc,
            spendpost_mb,
            spendpost_pl,
            spendpost_tox,
            spendpost_ur,
            spendpost_vir,
            spendpost_med,
            spendpost_mon,
            spendpost_exam,
            transpost_exam,
            waitn,
            trans,
            xmatch,
            spendpost_blg,
        ],
        keys=[
            "empfaenger",
            "organ_entnahme_niere",
            "spender_postmortem",
            "empfaenger_dringlichkeit",
            *("empfaenger_immunologie_" + newnames),
            "empfaenger_virologie",
            "followup_niere",
            "followup_niere_medikation",
            "spender_postmortem_diagnosen",
            "spender_postmortem_labor_blutgase",
            "spender_postmortem_labor_hla",
            "spender_postmortem_labor_klinische_chemie",
            "spender_postmortem_labor_mikrobiologie",
            "spender_postmortem_labor_pathologie",
            "spender_postmortem_labor_toxikologie",
            "spender_postmortem_labor_urin",
            "spender_postmortem_labor_virologie",
            "spender_postmortem_medikation",
            "spender_postmortem_monitoring",
            "spender_postmortem_untersuchungen",
            "transplantation_postop_untersuchung",
            "warteliste_niere",
            "transplantation",
            "spender_postmortem_labor_crossmatch",
            "spender_postmortem_labor_blutgruppe",
        ],
        axis=1,
        names=["dataset", "column"],
    )
    .sort_index(axis=0)
    .sort_index(axis=1)
)
display(
    Markdown(
        f"- **Final consolidated shape**: {consolidated.shape[0]} rows × {consolidated.shape[1]} columns"
    )
)
display(
    Markdown(
        f"- **Memory usage**: {consolidated.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
    )
)
display(
    Markdown(
        f"- **Number of datasets joined**: {consolidated.columns.get_level_values('dataset').nunique()}"
    )
)
display(
    Markdown(
        f"- **Null values**: {consolidated.isna().sum().sum()} ({100*consolidated.isna().sum().sum()/(consolidated.shape[0]*consolidated.shape[1]):.1f}%)"
    )
)

In [ ]:
display(Markdown("### Data Coverage by Dataset"))
coverage = (
    (~consolidated.isna())
    .reset_index(drop=True)
    .T.reset_index(drop=False)
    .drop(columns="column")
    .groupby("dataset")
    .apply(lambda col: col.any(axis=0), include_groups=False)
    .sum(axis=1)
    .sort_values()
    .to_frame("Transplants with Data")
)
display(coverage)

In [ ]:
consolidated.to_parquet(output_data)

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "util": util,
        "display_util": display_util,
    }
)